In [70]:
import numpy as np
import pandas as pd
from pathlib import Path

In [71]:
# Create CSV mapping ISO 3-letter codes to country names
#  (converts copy-pasted Wikipedia table into useful CSV)

# iso_df = pd.read_csv('iso_name_to_iso3c.csv', header=None)
# for index, row in iso_df.iterrows():
#     split = row[0].split('\xa0\xa0')
#     iso3c = split[0]
#     iso_name = split[1]
#     iso_df.at[index, 'iso3c'] = iso3c
#     iso_df.at[index, 'iso_name'] = iso_name
# iso_df = iso_df.drop(columns=[0])
# iso_df.to_csv('iso3c_to_iso_name.csv', index=False)
# iso_df

In [72]:
##############
### SCHEMA ###
##############

# - output/*/allocations/allocations_wide.csv has pathways of sorts
#   - has pathway for various emissions categories per ISO region
#   - want to map to IMAGE regions
#   - approach: ISO3C --> country name --> IMAGE region
# (idk what each emissions category is)
### 

In [73]:
image_mapping_table = pd.read_csv('IMAGE_region_country_mapping.csv')

IMAGE_COUNTRY_MAP = {}
for _, row in image_mapping_table.iterrows():
    region = row['Region']
    countries = row['Countries'].split(', ')
    countries = [c.split('(')[0].strip() for c in countries]

    for c in countries:
        IMAGE_COUNTRY_MAP[c] = region

In [74]:
iso3c_mapping_table = pd.read_csv('iso3c_to_iso_name.csv')
ISO3C_TO_ISO_NAME = {
    row['iso3c']: row['iso_name'] for _, row in iso3c_mapping_table.iterrows()
}

In [75]:
target = 'pathway'
OUT_PATH = Path(f"../output/primap-202503_wdi-2025_un-owid-2025_unu-wider-2025_melo-2026_{target}{'_exponential_decay' if target=='rcb_pathways' else ''}_all-ghg/")
RESULTS_PATH = OUT_PATH / 'allocations' / f'reference_pathway_allocations_{target}'

In [76]:
results = pd.read_csv(RESULTS_PATH /  'allocations_wide.csv')

In [77]:
# How many countries are unmapped or incorrectly mapped?
failed_countries = results[results['iso3c'].map(ISO3C_TO_ISO_NAME).map(IMAGE_COUNTRY_MAP).isna()]['iso3c'].unique()

for c in failed_countries:
    print(f"Failed to map {c} --> {ISO3C_TO_ISO_NAME.get(c, 'Unknown Country Name')}")

# NOTE for future: I fixed most of these by adjusting the `iso3c_to_iso_name` mapping to use IMAGE 
#  names instead of ISO names (e.g. 'United Kingdom of Great Britain and Northern Ireland' --> 'United Kingdom')

# Left with just 'ROW', which I imagine is "Rest of World" and thus unmappable to an IMAGE region.
# NOTE: Decided to drop ROW from the dataset.
results = results[results['iso3c'] != 'ROW']

Failed to map ROW --> Unknown Country Name


In [78]:
if 'region' not in results.columns:
    # surely mr. gpt can't be right that this is the cleanest way to add a column and change the order this way at the same time
    new_columns = results.columns.tolist()
    new_columns.insert(new_columns.index('iso3c') + 1, 'region')
    results['region'] = results['iso3c'].map(ISO3C_TO_ISO_NAME).map(IMAGE_COUNTRY_MAP)
    results = results[new_columns]

results.head(3)

,source-id,allocation-folder,emissions-source,gdp-source,population-source,gini-source,emission-category,target-source,data-type,approach-short,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,36.969249,36.645753,36.488582,36.135671,36.119341,36.103011,36.086681,36.02786,35.852762,35.677665
1,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-y2015-rw0.0-cw0.0-cy2050,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
# Group by region and approach-short, then sum year columns and keep other columns
year_columns = [col for col in results.columns if col.isdigit()]
non_year_columns = [col for col in results.columns if col not in year_columns and col not in ['region', 'approach-short']]

agg_dict = {col: 'sum' for col in year_columns}
agg_dict.update({col: 'first' for col in non_year_columns})

results_aggregated = results.groupby(['region', 'approach-short']).agg(agg_dict).reset_index()
results_aggregated = results_aggregated[new_columns]    # Reorder columns to match original order
results_aggregated = results_aggregated.drop(columns=['iso3c'])    # Drop iso3c column since it's no longer relevant after aggregation

results_aggregated.head(3)

,source-id,allocation-folder,emissions-source,gdp-source,population-source,gini-source,emission-category,target-source,data-type,approach-short,...,2091,2092,2093,2094,2095,2096,2097,2098,2099,2100
0,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-Adj-Gini-y2015-rw0.0-cw1.0-floor7500-gini...,...,9.860970,9.774757,9.732870,9.638818,9.634466,9.630114,9.625762,9.610086,9.563422,9.516758
1,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-Adj-Gini-y2015-rw0.5-cw0.5-hr2000-floor75...,...,13.468950,13.351190,13.293976,13.165508,13.159563,13.153619,13.147674,13.126262,13.062522,12.998783
2,primap-202503_wdi-2025_un-owid-2025_unu-wider-...,reference_pathway_allocations_pathway,primap-202503,wdi-2025,un-owid-2025,unu-wider-2025,all-ghg,pathway,absolute,CPCC-Adj-y2015-rw0.5-cw0.5-hr2000-cy2050,...,22.079352,21.886305,21.792513,21.581913,21.572168,21.562423,21.552678,21.517577,21.413087,21.308598


In [ ]:
results.to_csv(RESULTS_PATH / 'allocations_wide_mapped.csv', index=False)
results_aggregated.to_csv(RESULTS_PATH /  'allocations_wide_aggregated.csv', index=False)
